# 실습 5. RAG Framework 종합 비교 — 같은 코퍼스·같은 질의

## 실습 목표

LlamaIndex / Haystack / DSPy를 **동일 질문**으로 돌려 답변·출처·지연을 한 표로 비교합니다
(교안 6장). (RAGFlow는 Docker 엔진 — `ragflow_demo/` 시연으로 대체)

- 세 프레임워크가 같은 코퍼스에서 같은 출처를 검색하는지
- 차이는 '코드 구조·추상화'에 있음을 확인

> `MLAPI_*` 필요. 세 프레임워크를 모두 돌리므로 수십 초 걸립니다.
> (노트북 자립을 위해 세 build 함수를 inline으로 정의합니다.)

## 1. 세 프레임워크 runner 정의 (실습 1~3의 build를 inline)

In [ ]:
from week5.day22._common import EVAL_QUESTION, SAMPLE_DOCS, MLAPI_BASE_URL, MLAPI_API_KEY, MLAPI_MODEL, EMB_MODEL, have_mlapi
import time

def run_llamaindex(q):
    from llama_index.core import VectorStoreIndex, Document, Settings
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    from llama_index.llms.openai_like import OpenAILike
    Settings.embed_model = HuggingFaceEmbedding(model_name=EMB_MODEL)
    Settings.llm = OpenAILike(model=MLAPI_MODEL, api_base=MLAPI_BASE_URL, api_key=MLAPI_API_KEY, is_chat_model=True, context_window=8192, temperature=0.2)
    idx = VectorStoreIndex.from_documents([Document(text=d["text"], metadata={"topic": d["topic"]}) for d in SAMPLE_DOCS])
    resp = idx.as_query_engine(similarity_top_k=3).query(q)
    return str(resp).strip(), [n.metadata.get("topic","?") for n in resp.source_nodes]

def run_haystack(q):
    from haystack import Pipeline, Document
    from haystack.document_stores.in_memory import InMemoryDocumentStore
    from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
    from haystack.components.embedders import SentenceTransformersTextEmbedder, SentenceTransformersDocumentEmbedder
    from haystack.components.builders import PromptBuilder
    from haystack.components.generators import OpenAIGenerator
    from haystack.utils import Secret
    store = InMemoryDocumentStore()
    de = SentenceTransformersDocumentEmbedder(model=EMB_MODEL); de.warm_up()
    store.write_documents(de.run([Document(content=d["text"], meta={"topic": d["topic"]}) for d in SAMPLE_DOCS])["documents"])
    p = Pipeline()
    p.add_component("te", SentenceTransformersTextEmbedder(model=EMB_MODEL))
    p.add_component("re", InMemoryEmbeddingRetriever(store, top_k=3))
    p.add_component("pr", PromptBuilder(template="문맥:{% for d in documents %}({{d.meta.topic}}) {{d.content}}\n{% endfor %}\n질문:{{question}}\n문맥에 근거해 답하라.", required_variables=["question","documents"]))
    p.add_component("llm", OpenAIGenerator(api_key=Secret.from_token(MLAPI_API_KEY), api_base_url=MLAPI_BASE_URL, model=MLAPI_MODEL, generation_kwargs={"temperature":0.2}))
    p.connect("te.embedding","re.query_embedding"); p.connect("re.documents","pr.documents"); p.connect("pr.prompt","llm.prompt")
    res = p.run({"te":{"text":q},"pr":{"question":q}}, include_outputs_from={"re"})
    return res["llm"]["replies"][0].strip(), [d.meta.get("topic","?") for d in res["re"]["documents"]]

def run_dspy(q):
    import dspy
    from sentence_transformers import SentenceTransformer
    emb = SentenceTransformer(EMB_MODEL); dvec = emb.encode([d["text"] for d in SAMPLE_DOCS], normalize_embeddings=True)
    hits = [SAMPLE_DOCS[i] for i in (dvec @ emb.encode([q],normalize_embeddings=True)[0]).argsort()[::-1][:3]]
    class GA(dspy.Signature):
        """문맥에 근거해 한국어로 간결히 답한다."""
        context=dspy.InputField(); question=dspy.InputField(); answer=dspy.OutputField()
    dspy.configure(lm=dspy.LM(f"openai/{MLAPI_MODEL}", api_base=MLAPI_BASE_URL, api_key=MLAPI_API_KEY, temperature=0.2, max_tokens=512))
    ctx = "\n".join(f"({d['topic']}) {d['text']}" for d in hits)
    pred = dspy.ChainOfThought(GA)(context=ctx, question=q)
    return pred.answer.strip(), [d["topic"] for d in hits]

print("runner 3종 준비 완료")

## 2. 동일 질의로 비교 (교안 6.3 실측)

In [ ]:
rows = []
for name, fn in [("LlamaIndex", run_llamaindex), ("Haystack", run_haystack), ("DSPy", run_dspy)]:
    t0 = time.time(); ans, srcs = fn(EVAL_QUESTION); dt = time.time()-t0
    rows.append((name, srcs, len(ans), dt, ans))

print(f"{'프레임워크':<12}{'출처(topic)':<28}{'답변길이':>6}{'지연(s)':>9}")
print("-"*60)
for name, srcs, alen, dt, _ in rows:
    print(f"{name:<12}{str(srcs):<28}{alen:>6}{dt:>9.1f}")
print("\n[답변 요지]")
for name, _, _, _, ans in rows:
    print(f"● {name}: {ans[:90]}…")

**포인트**

- 세 프레임워크 모두 같은 코퍼스에서 **동일 출처(파인튜닝·RAG·평가)** 를 검색
- 즉 검색 품질은 프레임워크가 아니라 **구성요소(임베딩)** 가 결정(Day21 교훈)
- 차이는 답변 스타일·지연·코드 구조 — LlamaIndex(장황), Haystack(간결·빠름), DSPy(중간)

# [TODO]

## [TODO] 1. 다른 질문으로 비교

`EVAL_QUESTION` 대신 **다른 질문**(예: "벡터 데이터베이스에는 어떤 종류가 있나?")으로 세
프레임워크를 비교하세요. (출처가 셋 다 같은지 확인)

In [ ]:
# [TODO] 1: 다른 질문으로 세 프레임워크 비교(출처만)
# [TODO] 여기에 코드를 작성하세요.


## [TODO] 2. 가장 빠른 프레임워크 찾기

위 `rows`(name, srcs, alen, dt, ans)에서 **지연(dt)이 가장 작은 프레임워크 이름**을 출력하세요.

In [ ]:
# [TODO] 2: 가장 빠른 프레임워크
# [TODO] 여기에 코드를 작성하세요.


## 실습 정리

- 같은 코퍼스·같은 임베딩이면 세 프레임워크의 검색 출처는 동일
- 검색 품질=구성요소(임베딩)가 결정, 프레임워크 차이는 코드 구조·운영 방식
- 선택은 우열이 아니라 '데이터·팀·운영 조건에 무엇이 맞나'(교안 6.4 가이드)